In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
import time
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, matthews_corrcoef
import joblib
import warnings

# Ignore all warnings
warnings.filterwarnings('ignore')

In [2]:
from preprocess import preprocess_data
start_time = time.time()
df = preprocess_data('cicids2017.csv')
end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.4f} seconds")
df.describe()

Execution time: 64.0857 seconds


,srcip,sport,dstip,dsport,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,...,Idle Std,Idle Max,Idle Min,sport_Well-known,sport_Registered,sport_Dynamic/private,dsport_Well-known,dsport_Registered,dsport_Dynamic/private,attack_type
count,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,...,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06,2.829385e+06
mean,5.552657e-01,6.275913e-01,3.669243e-01,1.231594e-01,1.232731e-01,3.806352e-05,3.562146e-05,4.260202e-05,2.467057e-05,8.368235e-03,...,6.555082e-03,7.249938e-02,6.603194e-02,1.597467e-01,2.766969e-01,5.635564e-01,7.779585e-01,1.128001e-01,1.092414e-01,1.913817e+00
std,2.840046e-01,3.402244e-01,2.486167e-01,2.789873e-01,2.805021e-01,3.412175e-03,3.417446e-03,7.748822e-04,3.453537e-03,2.890179e-02,...,5.987095e-02,2.030999e-01,1.947365e-01,3.663711e-01,4.473654e-01,4.959442e-01,4.156190e-01,3.163484e-01,3.119419e-01,4.584751e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.876999e-01,5.000381e-01,1.783434e-01,8.087282e-04,1.400000e-06,4.550460e-06,3.425573e-06,9.302326e-07,3.051325e-09,2.417405e-04,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
50%,5.804814e-01,7.773861e-01,2.876999e-01,1.220722e-03,2.611666e-04,4.550460e-06,6.851145e-06,4.806202e-06,1.876565e-07,1.490733e-03,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
75%,8.453777e-01,8.913710e-01,4.531911e-01,6.759747e-03,2.684524e-02,1.820184e-05,1.370229e-05,1.449612e-05,7.353692e-07,3.263497e-03,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,5.000000e+00


In [11]:
df.shape

(2829385, 87)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2829385 entries, 0 to 2829384
Data columns (total 87 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   srcip                        float64
 1   sport                        float64
 2   dstip                        float64
 3   dsport                       float64
 4   Flow Duration                float64
 5   Total Fwd Packets            float64
 6   Total Backward Packets       float64
 7   Total Length of Fwd Packets  float64
 8   Total Length of Bwd Packets  float64
 9   Fwd Packet Length Max        float64
 10  Fwd Packet Length Min        float64
 11  Fwd Packet Length Mean       float64
 12  Fwd Packet Length Std        float64
 13  Bwd Packet Length Max        float64
 14  Bwd Packet Length Min        float64
 15  Bwd Packet Length Mean       float64
 16  Bwd Packet Length Std        float64
 17  Flow Bytes/s                 float64
 18  Flow Packets/s               float64
 19  

In [15]:
df['srcip'].head()

0    0.991155
1    0.080083
2    0.080083
3    0.869637
4    0.605986
Name: srcip, dtype: float64

In [17]:
import os
import joblib

label_encoder_path = 'label_encoder.pkl'
if os.path.exists(label_encoder_path):
    label_encoder = joblib.load(label_encoder_path)
    # Create a mapping: encoded integer -> attack category
    class_mapping = {index: label for index, label in enumerate(label_encoder.classes_)}
    print("Encoded class mapping:")
    for key, value in class_mapping.items():
        print(f"{key}: {value}")
else:
    print("Label encoder file not found. Run preprocess_data() first to generate it.")

Encoded class mapping:
0: Brute Force
1: Denial of Service (DoS/DDoS and Botnet)
2: Normal
3: Reconnaissance
4: Remote to Local (R2L)
5: Web Attack


In [19]:
# Separate features and target
X = df.drop('attack_type', axis=1)  # Features
y = df['attack_type']  # Target

# Step 1: Split the data into 70% training and 30% temporary set (for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Step 2: Split the temporary set into 50% validation and 50% test (15% each of the original dataset)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Print the shapes of the resulting splits
print("Shape of X_train:", X_train.shape)  # 70% of the data
print("Shape of X_val:", X_val.shape)      # 15% of the data
print("Shape of X_test:", X_test.shape)    # 15% of the data
print("Shape of y_train:", y_train.shape)
print("Shape of y_val:", y_val.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (1980569, 86)
Shape of X_val: (424408, 86)
Shape of X_test: (424408, 86)
Shape of y_train: (1980569,)
Shape of y_val: (424408,)
Shape of y_test: (424408,)


In [20]:
y_train.value_counts()

attack_type
2    1590881
1     267201
3     111251
0       9685
5       1526
4         25
Name: count, dtype: int64

In [ ]:
class_mapping= {0: 'Brute Force',
                1: 'Denial of Service (DoS/DDoS and Botnet)',
                2: 'Normal',
                3: 'Reconnaissance',
                4: 'Remote to Local (R2L)',
                5: 'Web Attack'}

# Train and validate XGBoost
def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test, model_filename):
    # Set up the plots
    plt.style.use('seaborn')
    fig = plt.figure(figsize=(15, 10))
    gs = plt.GridSpec(2, 2, figure=fig)
    
    # Batch processing plot
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, :])
    
    batch_sizes = []
    processed_samples = []
    current_metrics = {'precision': [], 'recall': [], 'f1': []}
    
    model = XGBClassifier(tree_method='hist', device='cuda', eval_metric="mlogloss",
                          random_state=42, num_classes=6, objective="multi:softmax", max_depth=15,
                          n_estimators=200, learning_rate=0.2)

    # Split data into batches
    batch_size = 1000
    n_batches = len(X_train) // batch_size
    
    start_time = time.time()
    
    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        
        X_batch = X_train[start_idx:end_idx]
        y_batch = y_train[start_idx:end_idx]
        
        # Update model with batch
        if i == 0:
            model.fit(X_batch, y_batch)
        else:
            model.fit(X_batch, y_batch, xgb_model=model)
        
        # Update tracking metrics
        batch_sizes.append(len(X_batch))
        processed_samples.append(end_idx)
        
        # Get current predictions and metrics
        y_pred = model.predict(X_val)
        report = classification_report(y_val, y_pred, output_dict=True)
        
        # Update metrics plot
        current_metrics['precision'].append(np.mean([report[str(i)]['precision'] for i in range(6)]))
        current_metrics['recall'].append(np.mean([report[str(i)]['recall'] for i in range(6)]))
        current_metrics['f1'].append(np.mean([report[str(i)]['f1-score'] for i in range(6)]))
        
        # Clear and redraw plots
        ax1.clear()
        ax2.clear()
        
        # Plot batch processing progress
        ax1.plot(processed_samples, 'b-', label='Processed Samples')
        ax1.set_title('Training Progress')
        ax1.set_xlabel('Batch Number')
        ax1.set_ylabel('Total Samples Processed')
        ax1.legend()
        
        # Plot metrics
        ax2.plot(current_metrics['precision'], 'r-', label='Precision')
        ax2.plot(current_metrics['recall'], 'g-', label='Recall')
        ax2.plot(current_metrics['f1'], 'b-', label='F1 Score')
        ax2.set_title('Training Metrics Progress')
        ax2.set_xlabel('Batch Number')
        ax2.set_ylabel('Score')
        ax2.legend()
        
        plt.tight_layout()
        plt.pause(0.1)
    
    training_time = time.time() - start_time
    
    # Final evaluation
    print("\nEvaluating on Validation Set:")
    evaluate_xgb(model, X_val, y_val)
    
    print("\nEvaluating on Test Set:")
    evaluate_xgb(model, X_test, y_test, ax3)
    
    # Save the model
    joblib.dump(model, model_filename)
    print(f"\nModel saved to {model_filename}")
    
    print('\nTraining Time: {:.2f} seconds'.format(training_time))
    plt.show()

def evaluate_xgb(model, X_eval, y_eval, ax=None):
    y_pred = model.predict(X_eval)
    
    print("\nClassification Report:")
    report = classification_report(y_eval, y_pred, digits=4, 
                                 target_names=[class_mapping[i] for i in range(len(class_mapping))])
    print(report)
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_eval, y_pred)
    print(cm)
    
    roc_auc = roc_auc_score(pd.get_dummies(y_eval), model.predict_proba(X_eval), multi_class="ovr")
    mcc = matthews_corrcoef(y_eval, y_pred)
    
    print("\nROC-AUC Score:", roc_auc)
    print("\nMatthews Correlation Coefficient:", mcc)
    
    if ax is not None:
        # Create final dashboard visualization
        report_dict = classification_report(y_eval, y_pred, output_dict=True)
        
        # Extract metrics for each class
        classes = list(class_mapping.values())
        metrics = ['precision', 'recall', 'f1-score']
        
        data = []
        for c in range(len(classes)):
            row = [report_dict[str(c)][m] for m in metrics]
            data.append(row)
        
        # Create heatmap
        ax.clear()
        sns.heatmap(data, annot=True, fmt='.3f', cmap='YlOrRd',
                    xticklabels=metrics, yticklabels=classes, ax=ax)
        ax.set_title('Final Classification Performance')
        plt.tight_layout()

Training XGBoost Classifier...

Evaluating on Validation Set:

Classification Report:
                                         precision    recall  f1-score   support

                            Brute Force     1.0000    0.9990    0.9995      2075
Denial of Service (DoS/DDoS and Botnet)     0.9997    0.9998    0.9998     57257
                                 Normal     1.0000    1.0000    1.0000    340904
                         Reconnaissance     0.9997    0.9996    0.9996     23840
                  Remote to Local (R2L)     1.0000    1.0000    1.0000         5
                             Web Attack     0.9878    0.9908    0.9893       327

                               accuracy                         0.9999    424408
                              macro avg     0.9979    0.9982    0.9980    424408
                           weighted avg     0.9999    0.9999    0.9999    424408


Confusion Matrix:
[[  2073      0      0      2      0      0]
 [     0  57246      7      2      0 

In [ ]:
from imblearn.over_sampling import SMOTE

# --------------------------
# Step 1: Duplicate minority classes in the training set
# --------------------------
# Combine the features and target into one DataFrame for easier manipulation
df_train = X_train.copy()
df_train['attack_type'] = y_train

# Separate the training data by class
df_train_0 = df_train[df_train['attack_type'] == 0]
df_train_1 = df_train[df_train['attack_type'] == 1]
df_train_2 = df_train[df_train['attack_type'] == 2]
df_train_3 = df_train[df_train['attack_type'] == 3]
df_train_4 = df_train[df_train['attack_type'] == 4]
df_train_5 = df_train[df_train['attack_type'] == 5]

# Duplicate:
# - For class 4: repeat each sample 40 times
# - For class 5: repeat each sample 3 times
df_train_4_dup = pd.concat([df_train_4] * 40, ignore_index=True)
df_train_5_dup = pd.concat([df_train_5] * 3, ignore_index=True)

# Combine the duplicated minority classes with the rest of the training data
df_train_aug = pd.concat([df_train_0, df_train_1, df_train_2, df_train_3,
                          df_train_4_dup, df_train_5_dup], ignore_index=True)

print("Training set shape after duplication:", df_train_aug.shape)
print("Class distribution after duplication:")
print(df_train_aug['attack_type'].value_counts())

# --------------------------
# Step 2: Apply SMOTE oversampling on the training set (after duplication)
# --------------------------
# Prepare features and target
X_train_aug = df_train_aug.drop(columns=['attack_type'])
y_train_aug = df_train_aug['attack_type']

# Define the SMOTE sampling strategy:
# Increase class 5 and class 4 to 20,000 samples
smote_strategy = {
    0: 20000,
    4: 20000,
    5: 20000
}

smote = SMOTE(sampling_strategy=smote_strategy, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_aug, y_train_aug)

print("Training set shape after SMOTE:", X_train_smote.shape)
print("Class distribution after SMOTE:")
unique, counts = np.unique(y_train_smote, return_counts=True)
print(dict(zip(unique, counts)))

# --------------------------
# Step 3: Train model on SMOTE oversampled data
# --------------------------
print("\nEvaluating model trained on SMOTE oversampled training data:")
train_xgboost(X_train_smote, y_train_smote, X_val, y_val, X_test, y_test, 'smote_xgb.pkl')

Training set shape after duplication: (1984596, 87)
Class distribution after duplication:
attack_type
2    1590881
1     267201
3     111251
0       9685
5       4578
4       1000
Name: count, dtype: int64
Training set shape after SMOTE: (2029333, 86)
Class distribution after SMOTE:
{0: 20000, 1: 267201, 2: 1590881, 3: 111251, 4: 20000, 5: 20000}

Evaluating model trained on SMOTE oversampled training data:

Evaluating on Validation Set:

Classification Report:
                                         precision    recall  f1-score   support

                            Brute Force     1.0000    0.9990    0.9995      2075
Denial of Service (DoS/DDoS and Botnet)     0.9997    0.9997    0.9997     57257
                                 Normal     1.0000    1.0000    1.0000    340904
                         Reconnaissance     0.9998    0.9997    0.9997     23840
                  Remote to Local (R2L)     1.0000    1.0000    1.0000         5
                             Web Attack     0.9

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
# --------------------------
# Step 4: Apply RandomUnderSampler to reduce classes 1, 2, 3 to 40,000 samples each
# --------------------------
# Define the undersampling strategy:
undersample_strategy = {
    1: 60000,
    2: 60000,
    3: 60000
}

undersampler = RandomUnderSampler(sampling_strategy=undersample_strategy, random_state=42)
X_train_final, y_train_final = undersampler.fit_resample(X_train_smote, y_train_smote)

print("Training set shape after undersampling:", X_train_final.shape)
print("Class distribution after undersampling:")
unique, counts = np.unique(y_train_final, return_counts=True)
print(dict(zip(unique, counts)))

# --------------------------
# Step 4: Train model on undersampled and SMOTE-oversampled data
# --------------------------
print("\nEvaluating model trained on final processed training data:")
train_xgboost(X_train_final, y_train_final, X_val, y_val, X_test, y_test, 'final_xgb.pkl')

Training set shape after undersampling: (240000, 86)
Class distribution after undersampling:
{0: 20000, 1: 60000, 2: 60000, 3: 60000, 4: 20000, 5: 20000}

Evaluating model trained on final processed training data:

Evaluating on Validation Set:

Classification Report:
                                         precision    recall  f1-score   support

                            Brute Force     1.0000    1.0000    1.0000      2075
Denial of Service (DoS/DDoS and Botnet)     0.9988    0.9997    0.9993     57257
                                 Normal     1.0000    0.9998    0.9999    340904
                         Reconnaissance     0.9999    0.9996    0.9997     23840
                  Remote to Local (R2L)     1.0000    1.0000    1.0000         5
                             Web Attack     0.9731    0.9969    0.9849       327

                               accuracy                         0.9998    424408
                              macro avg     0.9953    0.9993    0.9973    424408


In [41]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

##############################################
# Model Saver Class (Local PC)
##############################################

class LocalModelSaver:
    def __init__(self, base_path='./models'):
        self.base_path = Path(base_path)
        self.base_path.mkdir(parents=True, exist_ok=True)

    def save_checkpoint(self, generator, discriminator,
                        opt_g, opt_d, epoch, metrics,
                        model_name='gan_model'):
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        save_dir = self.base_path / model_name / timestamp
        save_dir.mkdir(parents=True, exist_ok=True)

        checkpoint = {
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_g_state_dict': opt_g.state_dict(),
            'optimizer_d_state_dict': opt_d.state_dict(),
            'metrics': metrics
        }

        torch.save(checkpoint, save_dir / 'checkpoint.pt')

        config = {
            'generator_config': {
                'latent_dim': generator.latent_dim,
                'label_dim': generator.label_dim
            },
            'epoch': epoch,
            'timestamp': timestamp
        }

        with open(save_dir / 'config.json', 'w') as f:
            json.dump(config, f, indent=4)

        print(f"Model saved successfully at: {save_dir}")
        return str(save_dir)

    def load_checkpoint(self, model_path, generator, discriminator,
                        opt_g=None, opt_d=None):
        checkpoint = torch.load(Path(model_path) / 'checkpoint.pt')
        generator.load_state_dict(checkpoint['generator_state_dict'])
        discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
        if opt_g is not None:
            opt_g.load_state_dict(checkpoint['optimizer_g_state_dict'])
        if opt_d is not None:
            opt_d.load_state_dict(checkpoint['optimizer_d_state_dict'])
        return checkpoint['epoch'], checkpoint['metrics']

    def save_quick(self, generator, discriminator, model_name='gan_model'):
        save_dir = self.base_path / model_name / 'quick_save'
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save(generator.state_dict(), save_dir / 'generator.pt')
        torch.save(discriminator.state_dict(), save_dir / 'discriminator.pt')
        print(f"Quick save completed at: {save_dir}")
        return str(save_dir)

##############################################
# Model Definitions
##############################################

class Generator(nn.Module):
    def __init__(self, latent_dim, label_dim, output_dim):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        self.label_dim = label_dim
        self.initial = nn.Linear(latent_dim + label_dim, 128)
        self.hidden1 = nn.Linear(128, 256)
        self.hidden2 = nn.Linear(256, 512)
        self.output = nn.Linear(512, output_dim)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, z, labels):
        if len(z.shape) == 1:
            z = z.unsqueeze(0)
        if len(labels.shape) == 1:
            labels = labels.unsqueeze(0)
        x = torch.cat((z, labels), dim=1)
        x = self.leaky(self.initial(x))
        x = self.leaky(self.hidden1(x))
        x = self.leaky(self.hidden2(x))
        return torch.sigmoid(self.output(x))

class Discriminator(nn.Module):
    def __init__(self, input_dim, label_dim):
        super(Discriminator, self).__init__()
        self.input_dim = input_dim
        self.label_dim = label_dim
        self.initial = nn.Linear(input_dim + label_dim, 512)
        self.hidden1 = nn.Linear(512, 256)
        self.hidden2 = nn.Linear(256, 128)
        self.output = nn.Linear(128, 1)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, x, labels):
        if len(x.shape) == 1:
            x = x.unsqueeze(0)
        if len(labels.shape) == 1:
            labels = labels.unsqueeze(0)
        x = torch.cat((x, labels), dim=1)
        x = self.leaky(self.initial(x))
        x = self.leaky(self.hidden1(x))
        x = self.leaky(self.hidden2(x))
        return self.output(x)

##############################################
# Helper Classes and Functions
##############################################

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False

    def __call__(self, current_loss):
        if self.best_loss is None:
            self.best_loss = current_loss
            return False
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

def evaluate_model(generator, discriminator, val_loader, device):
    generator.eval()
    discriminator.eval()
    total_g_loss = 0.0
    total_d_loss = 0.0
    batches = 0

    with torch.no_grad():
        for real_data, real_labels in val_loader:
            current_batch_size = real_data.size(0)
            real_data = real_data.to(device)
            real_labels = real_labels.to(device)

            z = torch.randn(current_batch_size, generator.latent_dim, device=device)
            fake_data = generator(z, real_labels)

            fake_output = discriminator(fake_data, real_labels)
            g_loss = F.binary_cross_entropy_with_logits(
                fake_output, torch.ones_like(fake_output, device=device) * 0.9
            )

            d_real = discriminator(real_data, real_labels)
            d_fake = discriminator(fake_data, real_labels)
            d_loss_real = F.binary_cross_entropy_with_logits(
                d_real, torch.ones_like(d_real, device=device) * 0.9
            )
            d_loss_fake = F.binary_cross_entropy_with_logits(
                d_fake, torch.zeros_like(d_fake, device=device) + 0.1
            )
            d_loss = (d_loss_real + d_loss_fake) / 2

            total_g_loss += g_loss.item()
            total_d_loss += d_loss.item()
            batches += 1

    avg_g_loss = total_g_loss / batches if batches > 0 else float('inf')
    avg_d_loss = total_d_loss / batches if batches > 0 else float('inf')
    return avg_g_loss, avg_d_loss

##############################################
# Training Functions
##############################################

def train_gan(generator, discriminator, train_loader, val_loader, device, model_saver,
              num_epochs=200, save_interval=200, patience=5, min_delta=0.001):

    generator = generator.to(device)
    discriminator = discriminator.to(device)

    optimizer_G = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    scaler = GradScaler()

    early_stopping = EarlyStopping(patience=patience, min_delta=min_delta)

    history = {
        'g_loss': [],
        'd_loss': [],
        'val_g_loss': [],
        'val_d_loss': []
    }

    best_g_loss = float('inf')

    for epoch in range(num_epochs):
        generator.train()
        discriminator.train()

        total_g_loss = 0
        total_d_loss = 0
        batches_processed = 0

        for real_data, real_labels in train_loader:
            current_batch_size = real_data.size(0)
            real_data = real_data.to(device)
            real_labels = real_labels.to(device)

            with autocast():
                z = torch.randn(current_batch_size, generator.initial.in_features - real_labels.size(1), device=device)
                fake_data = generator(z, real_labels)
                d_real = discriminator(real_data, real_labels)
                d_fake = discriminator(fake_data.detach(), real_labels)

                d_loss_real = F.binary_cross_entropy_with_logits(
                    d_real, torch.ones_like(d_real, device=device) * 0.9)
                d_loss_fake = F.binary_cross_entropy_with_logits(
                    d_fake, torch.zeros_like(d_fake, device=device) + 0.1)
                d_loss = (d_loss_real + d_loss_fake) / 2

            optimizer_D.zero_grad(set_to_none=True)
            scaler.scale(d_loss).backward()
            scaler.step(optimizer_D)
            scaler.update()

            total_d_loss += d_loss.item()

            for _ in range(3):
                optimizer_G.zero_grad(set_to_none=True)
                with autocast():
                    z = torch.randn(current_batch_size, generator.initial.in_features - real_labels.size(1), device=device)
                    fake_data = generator(z, real_labels)
                    fake_output = discriminator(fake_data, real_labels)
                    g_loss = F.binary_cross_entropy_with_logits(
                        fake_output, torch.ones_like(fake_output, device=device) * 0.9)
                scaler.scale(g_loss).backward()
                scaler.step(optimizer_G)
                scaler.update()
                total_g_loss += g_loss.item()

            batches_processed += 1

        avg_g_loss = total_g_loss / (batches_processed * 3)
        avg_d_loss = total_d_loss / batches_processed

        history['g_loss'].append(avg_g_loss)
        history['d_loss'].append(avg_d_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] G_loss: {avg_g_loss:.4f} D_loss: {avg_d_loss:.4f}")

        if (epoch + 1) % save_interval == 0:
            val_g_loss, val_d_loss = evaluate_model(generator, discriminator, val_loader, device)

            metrics = {
                'g_loss': avg_g_loss,
                'd_loss': avg_d_loss,
                'val_g_loss': val_g_loss,
                'val_d_loss': val_d_loss
            }

            save_path = model_saver.save_checkpoint(
                generator, discriminator,
                optimizer_G, optimizer_D,
                epoch, metrics
            )

            if val_g_loss < best_g_loss:
                best_g_loss = val_g_loss

            if early_stopping(val_g_loss):
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                break

    final_path = model_saver.save_quick(generator, discriminator)
    return history

##############################################
# Data Preparation Functions
##############################################

def prepare_data(X, y, batch_size=1000):
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values

    encoder = OneHotEncoder(sparse_output=False)
    y_one_hot = encoder.fit_transform(y.reshape(-1, 1))

    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y_one_hot, dtype=torch.float32)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    return loader, encoder, y_one_hot.shape[1]

def generate_synthetic_data(generator, num_samples, label_dim, device, X_train, y_train):
    # Ensure generator is on the proper device.
    generator = generator.to(device)
    generator.eval()
    synthetic_data = []
    synthetic_labels = []

    with torch.no_grad():
        for class_idx in range(label_dim):
            labels = torch.zeros(num_samples, label_dim, device=device)
            labels[:, class_idx] = 1

            z = torch.randn(num_samples, generator.initial.in_features - label_dim, device=device)
            fake_data = generator(z, labels).cpu().numpy()

            synthetic_data.append(fake_data)
            synthetic_labels.append(np.full(num_samples, class_idx))

    X_synthetic = pd.DataFrame(np.vstack(synthetic_data), columns=X_train.columns)
    
    if hasattr(y_train, 'columns'):
        label_columns = y_train.columns
    elif hasattr(y_train, 'name') and y_train.name is not None:
        label_columns = [y_train.name]
    else:
        label_columns = ['target']
    
    y_synthetic = pd.DataFrame(np.hstack(synthetic_labels), columns=label_columns)
    
    return X_synthetic, y_synthetic

##############################################
# Main Function
##############################################

def main():
    torch.manual_seed(42)
    np.random.seed(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Initialize the model saver
    model_saver = LocalModelSaver()

    batch_size = 1000
    latent_dim = 200
    num_epochs = 200

    print(f"Training data shapes - X: {X_train_final.shape}, y: {y_train_final.shape}")

    # Prepare data
    train_loader, encoder, num_classes = prepare_data(X_train_final, y_train_final, batch_size)
    val_loader, _, _ = prepare_data(X_val, y_val, batch_size)

    print(f"Number of features: {X_train_final.shape[1]}")
    print(f"Number of classes: {num_classes}")
    print(f"Latent dimension: {latent_dim}")

    generator = Generator(latent_dim, num_classes, X_train_final.shape[1])
    discriminator = Discriminator(X_train_final.shape[1], num_classes)

    print("\nGenerator architecture:")
    print(generator)
    print("\nDiscriminator architecture:")
    print(discriminator)

    history = train_gan(generator, discriminator, train_loader, val_loader,
                        device, model_saver, num_epochs=num_epochs, save_interval=10,
                        patience=40, min_delta=0.0005)

    X_synthetic, y_synthetic = generate_synthetic_data(generator, 30000, num_classes, device, X_train_final, y_train_final)

    return generator, discriminator, history, X_synthetic, y_synthetic

# Assuming X_train_final, y_train_final, X_val, y_val are defined in the environment
generator, discriminator, history, X_synthetic, y_synthetic = main()


Using device: cuda
Training data shapes - X: (240000, 86), y: (240000,)
Number of features: 86
Number of classes: 6
Latent dimension: 200

Generator architecture:
Generator(
  (initial): Linear(in_features=206, out_features=128, bias=True)
  (hidden1): Linear(in_features=128, out_features=256, bias=True)
  (hidden2): Linear(in_features=256, out_features=512, bias=True)
  (output): Linear(in_features=512, out_features=86, bias=True)
  (leaky): LeakyReLU(negative_slope=0.2)
)

Discriminator architecture:
Discriminator(
  (initial): Linear(in_features=92, out_features=512, bias=True)
  (hidden1): Linear(in_features=512, out_features=256, bias=True)
  (hidden2): Linear(in_features=256, out_features=128, bias=True)
  (output): Linear(in_features=128, out_features=1, bias=True)
  (leaky): LeakyReLU(negative_slope=0.2)
)
Epoch [1/200] G_loss: 0.7907 D_loss: 0.6549
Epoch [2/200] G_loss: 1.1907 D_loss: 0.5321
Epoch [3/200] G_loss: 1.4182 D_loss: 0.4675
Epoch [4/200] G_loss: 1.4048 D_loss: 0.453

AttributeError: 'Series' object has no attribute 'columns'

In [53]:
def generate_synthetic_data(generator, num_samples, label_dim, device, X_train, y_train):
    # Ensure generator is on the proper device.
    generator = generator.to(device)
    generator.eval()
    synthetic_data = []
    synthetic_labels = []

    with torch.no_grad():
        for class_idx in range(label_dim):
            labels = torch.zeros(num_samples, label_dim, device=device)
            labels[:, class_idx] = 1

            z = torch.randn(num_samples, generator.initial.in_features - label_dim, device=device)
            fake_data = generator(z, labels).cpu().numpy()

            synthetic_data.append(fake_data)
            synthetic_labels.append(np.full(num_samples, class_idx))

    X_synthetic = pd.DataFrame(np.vstack(synthetic_data), columns=X_train.columns)
    
    if hasattr(y_train, 'columns'):
        label_columns = y_train.columns
    elif hasattr(y_train, 'name') and y_train.name is not None:
        label_columns = [y_train.name]
    else:
        label_columns = ['target']
    
    y_synthetic = pd.DataFrame(np.hstack(synthetic_labels), columns=label_columns)
    
    return X_synthetic, y_synthetic

In [55]:
# Define hyperparameters and data (assuming these are defined somewhere)
latent_dim = 200
num_classes = 6  
input_dim = X_train_final.shape[1] 

# Create the generator architecture (must match training)
generator = Generator(latent_dim, num_classes, input_dim)

# Load the saved generator weights (update the path as needed)
generator_path = Path('./models/gan_model/quick_save/generator.pt')
generator.load_state_dict(torch.load(generator_path, map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu")))
generator.eval()

# Use the synthetic data generation function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_synthetic, y_synthetic = generate_synthetic_data(generator, 30000, num_classes, device, X_train_final, y_train_final)

print("Synthetic data generated successfully!")

Synthetic data generated successfully!


In [ ]:
print(f"Synthetic data shape: {X_synthetic.shape}")
print("Training XGBoost on synthetic data...")
train_xgboost(X_synthetic, y_synthetic, X_val, y_val, X_test, y_test, 'synthetic_xgb.pkl')

Synthetic data shape: (180000, 86)
Training XGBoost on synthetic data...

Evaluating on Validation Set:

Classification Report:
                                         precision    recall  f1-score   support

                            Brute Force     0.0593    0.7171    0.1096      2075
Denial of Service (DoS/DDoS and Botnet)     0.4053    0.6273    0.4924     57257
                                 Normal     1.0000    0.4774    0.6463    340904
                         Reconnaissance     0.1847    0.9824    0.3110     23840
                  Remote to Local (R2L)     0.0006    1.0000    0.0012         5
                             Web Attack     0.0221    0.8563    0.0432       327

                               accuracy                         0.5275    424408
                              macro avg     0.2787    0.7767    0.2673    424408
                           weighted avg     0.8686    0.5275    0.6036    424408


Confusion Matrix:
[[  1488    584      0      3      0    

In [75]:
def augment_data(X_train, y_train, X_synthetic, y_synthetic):
    """
    Augments the original training data with generated synthetic data.

    Parameters:
        X_train (pd.DataFrame): Original training features.
        y_train (pd.Series or np.array): Original training labels.
        X_synthetic (np.array or pd.DataFrame): Synthetic features generated by the CGAN.
        y_synthetic (np.array or pd.Series): Synthetic labels corresponding to X_synthetic.

    Returns:
        X_augmented (pd.DataFrame): Augmented training features.
        y_augmented (np.array): Augmented training labels.
    """
    # If X_synthetic is not a DataFrame, convert it using X_train's columns
    if not isinstance(X_synthetic, pd.DataFrame):
        X_synthetic = pd.DataFrame(X_synthetic, columns=X_train.columns)

    # Concatenate feature DataFrames along rows
    X_augmented = pd.concat([X_train, X_synthetic], axis=0)
    
    # Ensure y_train and y_synthetic are 1D arrays before concatenating
    y_train = np.array(y_train).flatten()
    y_synthetic = np.array(y_synthetic).flatten()
    y_augmented = np.hstack((y_train, y_synthetic))

    return X_augmented, y_augmented

# Use the original DataFrame rather than converting to NumPy arrays
X_augmented, y_augmented = augment_data(X_train_smote, y_train_smote, X_synthetic, y_synthetic)

print(f"Original training data shape: {X_train.shape}")
print(f"Augmented training data shape: {X_augmented.shape}")

Original training data shape: (1980569, 86)
Augmented training data shape: (2209333, 86)


In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from tqdm import tqdm
import multiprocessing as mp
from multiprocessing import Pool
import os

# Optimizing the CGANEvaluator class
class CGANEvaluator:
    def __init__(self, generator, original_data, original_labels, device, n_jobs=-1):
        self.generator = generator
        self.original_data = original_data
        self.original_labels = original_labels
        self.device = device
        self.n_jobs = n_jobs  # Number of jobs for parallel execution
        self.scaler = StandardScaler()
        self.original_data_scaled = self.scaler.fit_transform(original_data)

    def generate_samples_for_class(self, class_idx, samples_per_class, unique_labels):
        """Generate synthetic samples for each class"""
        labels = torch.zeros(samples_per_class, len(unique_labels), device=self.device)
        labels[:, class_idx] = 1

        # Generate latent vectors
        z = torch.randn(samples_per_class,
                        self.generator.latent_dim,
                        device=self.device)

        # Generate fake data
        fake_data = self.generator(z, labels).cpu().numpy()

        synthetic_labels = np.full(samples_per_class, class_idx)
        return fake_data, synthetic_labels

    def generate_samples(self, samples_per_class):
        """Generate synthetic samples for each class in parallel"""
        self.generator.eval()
        unique_labels = np.unique(self.original_labels)

        # Adjust number of jobs
        if self.n_jobs == -1:
            self.n_jobs = os.cpu_count()  # Use all available cores
        elif self.n_jobs <= 0:
            self.n_jobs = 1  # Use at least 1 process

        # Generate in batches
        batch_size = 5000  # Process in batches to save memory and reduce overhead
        all_synthetic_data = []
        all_synthetic_labels = []

        for start in range(0, samples_per_class, batch_size):
            end = min(start + batch_size, samples_per_class)
            with Pool(processes=self.n_jobs) as pool:
                results = pool.starmap(self.generate_samples_for_class,
                                       [(class_idx, end - start, unique_labels) for class_idx in unique_labels])

            batch_data = np.vstack([result[0] for result in results])
            batch_labels = np.hstack([result[1] for result in results])

            all_synthetic_data.append(batch_data)
            all_synthetic_labels.append(batch_labels)

        # Combine all batches
        synthetic_data = np.vstack(all_synthetic_data)
        synthetic_labels = np.hstack(all_synthetic_labels)

        # Scale the synthetic data
        self.synthetic_data_scaled = self.scaler.transform(synthetic_data)

        return synthetic_data, synthetic_labels

    def calculate_distribution_metrics_for_feature(self, feature_idx):
        """Calculate KS and Wasserstein for each feature"""
        ks_stat, _ = ks_2samp(self.original_data_scaled[:, feature_idx],
                               self.synthetic_data_scaled[:, feature_idx])
        w_dist = wasserstein_distance(self.original_data_scaled[:, feature_idx],
                                      self.synthetic_data_scaled[:, feature_idx])
        return ks_stat, w_dist

    def calculate_distribution_metrics(self):
        """Calculate distribution similarity metrics in parallel"""
        with Pool(processes=self.n_jobs) as pool:
            results = pool.map(self.calculate_distribution_metrics_for_feature, range(self.original_data.shape[1]))

        ks_stats, wasserstein_stats = zip(*results)
        metrics = {
            'ks_test_mean': np.mean(ks_stats),
            'ks_test_std': np.std(ks_stats),
            'wasserstein_mean': np.mean(wasserstein_stats),
            'wasserstein_std': np.std(wasserstein_stats),
            'original_silhouette': silhouette_score(self.original_data_scaled, self.original_labels),
            'synthetic_silhouette': silhouette_score(self.synthetic_data_scaled, self.synthetic_labels)
        }

        return metrics

    def visualize_distributions(self):
        """Create visualizations comparing original and synthetic data"""
        tsne = TSNE(n_components=2, random_state=42, n_jobs=1)  # Use only 1 CPU for TSNE

        # Combine data for t-SNE
        combined_data = np.vstack([self.original_data_scaled, self.synthetic_data_scaled])
        combined_labels = np.hstack([self.original_labels, self.synthetic_labels])
        data_type = np.array(['Original'] * len(self.original_labels) + ['Synthetic'] * len(self.synthetic_labels))

        tsne_results = tsne.fit_transform(combined_data)

        # Create figure with subplots
        fig = plt.figure(figsize=(20, 10))

        # t-SNE plot by class
        plt.subplot(1, 2, 1)
        scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1],
                              c=combined_labels, cmap='tab10', alpha=0.6)
        plt.colorbar(scatter)
        plt.title('t-SNE Visualization by Class')

        # t-SNE plot by data type
        plt.subplot(1, 2, 2)
        for data_type_name in ['Original', 'Synthetic']:
            mask = data_type == data_type_name
            plt.scatter(tsne_results[mask, 0], tsne_results[mask, 1], label=data_type_name, alpha=0.6)
        plt.legend()
        plt.title('t-SNE Visualization by Data Type')

        plt.tight_layout()
        plt.show()

        # Feature distributions
        num_features = min(6, self.original_data.shape[1])  # Show first 6 features
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.ravel()

        for i in range(num_features):
            sns.kdeplot(data=self.original_data[:, i], ax=axes[i],
                        label='Original', alpha=0.6)
            sns.kdeplot(data=self.synthetic_data[:, i], ax=axes[i],
                        label='Synthetic', alpha=0.6)
            axes[i].set_title(f'Feature {i+1} Distribution')
            axes[i].legend()

        plt.tight_layout()
        plt.show()

    def class_balance_analysis(self):
        """Analyze class distribution balance"""
        orig_class_counts = pd.Series(self.original_labels).value_counts()
        syn_class_counts = pd.Series(self.synthetic_labels).value_counts()

        # Plot class distributions with rotated labels and adjusted figure size
        plt.figure(figsize=(15, 8))
        
        x = np.arange(len(orig_class_counts))
        width = 0.35
        
        plt.bar(x - width/2, orig_class_counts, width, label='Original')
        plt.bar(x + width/2, syn_class_counts, width, label='Synthetic')
        
        plt.xlabel('Class')
        plt.ylabel('Count')
        plt.title('Class Distribution Comparison')
        plt.legend()
        
        # Rotate x-axis labels for better readability
        plt.xticks(x, [class_mapping[i] for i in range(len(class_mapping))], rotation=45, ha='right')
        
        # Adjust layout to prevent label cutoff
        plt.tight_layout()
        plt.show()
        
        return {
            'original_distribution': orig_class_counts,
            'synthetic_distribution': syn_class_counts
        }

# Main function to evaluate the performance of the CGAN model
def evaluate_cgan(generator, original_data, original_labels, device, samples_per_class=50000, n_jobs=-1):
    """Main function to evaluate CGAN performance"""
    evaluator = CGANEvaluator(generator, original_data, original_labels, device, n_jobs=n_jobs)

    # Generate synthetic samples
    print("Generating synthetic samples...")
    synthetic_data, synthetic_labels = evaluator.generate_samples(samples_per_class)

    # Calculate metrics
    print("\nCalculating distribution metrics...")
    metrics = evaluator.calculate_distribution_metrics()

    # Print metrics
    print("\nEvaluation Metrics:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")

    # Generate visualizations
    print("\nGenerating visualizations...")
    evaluator.visualize_distributions()

    # Analyze class balance
    print("\nAnalyzing class balance...")
    balance_metrics = evaluator.class_balance_analysis()

    return synthetic_data, synthetic_labels, metrics, evaluator

# Set multiprocessing start method to 'spawn' (this is crucial for CUDA)
if __name__ == '__main__':
    mp.set_start_method('spawn', force=True)

    # Ensure that at least one process is used and we don't use zero processes
    num_processes = os.cpu_count() or 1  # This ensures at least one process is used

    # Your code where you run evaluation
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Proceed with your CGAN evaluation
    X_synthetic, y_synthetic, metrics, evaluator = evaluate_cgan(
        generator=generator,
        original_data=X_train_smote,  # Your original training data
        original_labels=y_train_smote,  # Your original training labels
        device=device,
        samples_per_class=20000
    )

Generating synthetic samples...


In [ ]:
# Function to train and validate the XGBoost model and save it to a file
def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test, model_filename):
    model = XGBClassifier(tree_method='hist', device='cuda', eval_metric="mlogloss",
                          random_state=42, num_classes=5, objective="multi:softmax", max_depth=15,
                          n_estimators=200, learning_rate=0.2)

    # Training the model without early stopping
    start_time = time.time()
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose = False)
    training_time = time.time() - start_time

    print("\nEvaluating on Validation Set:")
    evaluate_xgb(model, X_val, y_val)

    print("\nEvaluating on Test Set:")
    evaluate_xgb(model, X_test, y_test)

    # Save the trained model to a file
    joblib.dump(model, model_filename)
    print(f"\nModel saved to {model_filename}")

    print('\nTraining Time: {:.2f} seconds'.format(training_time))

# Function to evaluate the model (for both validation and test sets)
def evaluate_xgb(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred, digits=4, target_names=[class_mapping[i] for i in range(len(class_mapping))]))

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_eval, y_pred)
    print(cm)

    print("\nROC-AUC Score:", roc_auc_score(pd.get_dummies(y_eval), model.predict_proba(X_eval), multi_class="ovr"))
    print("\nMatthews Correlation Coefficient:", matthews_corrcoef(y_eval, y_pred))

print("Training XGBoost on augmented data...")
train_xgboost(X_augmented, y_augmented, X_val, y_val, X_test, y_test,'cic_xgb.pkl')

In [73]:
def save_test_data(X_test, y_test, filename="test_data.csv"):
    """
    Saves the X_test features and y_test labels as a single CSV file.

    Parameters:
        X_test (pd.DataFrame): Test features.
        y_test (pd.Series or np.array): Test labels.
        filename (str): The name of the output CSV file.
    """
    # If y_test is a NumPy array, convert it to a DataFrame
    if not isinstance(y_test, pd.Series):
        y_test = pd.Series(y_test, name='Label')

    # Concatenate X_test and y_test along columns
    test_data = pd.concat([X_test, y_test], axis=1)

    # Save the DataFrame to a CSV file
    test_data.to_csv(filename, index=False)
    print(f"Test data saved to {filename}")

save_test_data(X_test, y_test, "cic_test.csv")

Test data saved to cic_test.csv
